**Theme:** 

*“A model is just a structured computation graph with registered parameters.”*

In [1]:
import torch
import torch.nn as nn



torch.__version__

'2.8.0+cu129'

# Simple Model and trainable parameters

## My first nn.Module

In [2]:
class SimpleModel(nn.Module):

    def __init__(self):
        super().__init__() 
        self.w = nn.Parameter(torch.tensor(2.0)) 
        self.b = nn.Parameter(torch.tensor(1.0)) 

    def forward(self,x): 
        return self.w*x + self.b 
    
model = SimpleModel() 

x = torch.tensor(3.0) 
y = model(x) 
print(y) 


tensor(7., grad_fn=<AddBackward0>)


- Why do we inherit from `nn.Module`?

- Why is w wrapped in `nn.Parameter`?

## Inspect trainable parameters

In [3]:
for name,param in model.named_parameters():
    print(name,param, param.requires_grad) 

w Parameter containing:
tensor(2., requires_grad=True) True
b Parameter containing:
tensor(1., requires_grad=True) True


- How many parameters are there?

- Why are they automatically tracked?

- What would happen if we used a plain tensor instead?

## Break the `__init__` on purpose

In [4]:
class SimpleModel(nn.Module):

    def __init__(self):
        super().__init__() 
        self.w = nn.Parameter(torch.tensor(2.0)) 
        self.b = nn.Parameter(torch.tensor(1.0)) 
        self.c = torch.tensor(5.0) 

    def forward(self,x): 
        return self.w*x + self.b 
    
model = SimpleModel() 

print(list(model.parameters()))


[Parameter containing:
tensor(2., requires_grad=True), Parameter containing:
tensor(1., requires_grad=True)]


- Why is `c` missing?

- Will `c` be updated during training?

## $Forward\neq Train$

In [5]:
y = model(x) 
y.backward() 

In [6]:
print(model.w.grad) 
print(model.b.grad) 

tensor(3.)
tensor(1.)


- Why does `backward()` work without an optimizer?

- What does the optimizer actually do then?

## Optimizer reality check

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1) 

optimizer.zero_grad()
y = model(x) 
loss = (y-10)**2 
loss.backward() 
optimizer.step()

print("w", model.w)
print("w", model.b) 

Parameter containing:
tensor(3.8000, requires_grad=True)
Parameter containing:
tensor(1.6000, requires_grad=True)


- What changed?

- What exactly did `optimizer.step()` do?

## Freeze parameters (Industry Concept)

In [8]:
model.w.requires_grad = False 

optimizer.zero_grad() 
y = model(x) 

loss = (y-10) ** 2 
loss.backward() 
optimizer.step() 

print('w', model.w) 
print('b', model.b) 

w Parameter containing:
tensor(3.8000)
b Parameter containing:
tensor(1.0000, requires_grad=True)


- Which parameter updated?

- Why is this important in transfer learning?

# Interaction

### 1️⃣ Why inherit from nn.Module

Professional explanation:

`nn.Module` is PyTorch’s base class that gives your model three critical capabilities:

- Parameter registration

- Automatic device movement (CPU/GPU)

- Recursive tracking of submodules

When we inherit from `nn.Module`:

- PyTorch knows what is part of the model

- It can find all parameters

- It can save/load weights

- It can move everything to GPU with .to(device)

Without `nn.Module`, PyTorch cannot treat our code as a model.
***

### 2️⃣ nn.Parameter vs Tensor

The exact rule is:

- Only tensors wrapped in `nn.Parameter` are treated as trainable model weights.

- Everything else is considered constant unless manually included.

This is why:

- `model.parameters()`

  - only returns `w` and `b`.

***
### 3️⃣ Why `c` disappears


**Important interview line:**

PyTorch only optimizes parameters that are registered inside `nn.Module`.

***

### 4️⃣ Why `backward()` works without optimizer



Idea:

- backward() computes gradients using the computation graph.
- It does not change any parameters.

Optimizer exists to:

- Take gradients from `.grad` and update `.data`

So:
```bash
loss.backward()  → computes gradients
optimizer.step() → applies gradients
```
They are separate on purpose.

### 5️⃣ What `optimizer.step()` really does




For SGD:

$𝑤 = 𝑤 − \eta⋅∇𝑤$

So `optimizer.step()` does:

- Use the gradients stored in .grad to modify the parameter values.

### 6️⃣ Freezing parameters (TRANSFER LEARNING — THIS IS IMPORTANT)



What freezing actually means:
```python
model.w.requires_grad = False
```
means:

- “Do NOT compute or update gradients for this parameter”

So:

- No `.grad`

- No update in `optimizer.step()`

***

#### Why this is used in Transfer Learning

In transfer learning:

- You start with a pretrained network

Early layers learned:

- edges

- textures

- shapes

These are general features
***
So we freeze them:
```python
for param in backbone.parameters():
    param.requires_grad = False
```
Then only train:

- The new classifier head

- Task-specific layers

This gives:

- Faster training

- Less overfitting

- Better performance on small datasets

📌 Interview answer:

- Freezing prevents pretrained knowledge from being destroyed while learning a new task.